# Import Required Libraries
Import necessary libraries including NLTK, scikit-learn, numpy, and sentence embedding libraries.

In [2]:
# Import necessary libraries
import nltk
from nltk.corpus import treebank
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np
from sentence_transformers import SentenceTransformer

/Users/larsheijnen/VSCode_S2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load and Explore the Treebank Corpus
Download and load the NLTK treebank corpus, explore its structure and contents.

In [3]:
# Download the treebank corpus
nltk.download('treebank')

# Load the treebank corpus
sentences = treebank.sents()
tagged_sentences = treebank.tagged_sents()

# Explore the structure and contents of the corpus
print(f"Number of sentences in the treebank corpus: {len(sentences)}")
print(f"First sentence: {sentences[0]}")
print(f"First tagged sentence: {tagged_sentences[0]}")

# Check the structure of a tagged sentence
print(f"Tagged words in the first sentence: {tagged_sentences[0]}")

[nltk_data] Downloading package treebank to
[nltk_data]     /Users/larsheijnen/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


Number of sentences in the treebank corpus: 3914
First sentence: ['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'Nov.', '29', '.']
First tagged sentence: [('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ'), (',', ','), ('will', 'MD'), ('join', 'VB'), ('the', 'DT'), ('board', 'NN'), ('as', 'IN'), ('a', 'DT'), ('nonexecutive', 'JJ'), ('director', 'NN'), ('Nov.', 'NNP'), ('29', 'CD'), ('.', '.')]
Tagged words in the first sentence: [('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ'), (',', ','), ('will', 'MD'), ('join', 'VB'), ('the', 'DT'), ('board', 'NN'), ('as', 'IN'), ('a', 'DT'), ('nonexecutive', 'JJ'), ('director', 'NN'), ('Nov.', 'NNP'), ('29', 'CD'), ('.', '.')]


# Extract Sentences and Check for Verbs
Extract sentences from the treebank corpus and create labels indicating whether each sentence contains a verb based on POS tags.

In [4]:
# Extract Sentences and Check for Verbs

# Create labels indicating whether each sentence contains a verb based on POS tags
def contains_verb(tagged_sentence):
    for word, tag in tagged_sentence:
        if tag.startswith('VB'):  # Tags for verbs start with 'VB'
            return 1
    return 0

# Generate labels for each sentence
labels = [contains_verb(sentence) for sentence in tagged_sentences]

# Display the first 10 sentences and their corresponding labels
for i in range(10):
    print(f"Sentence: {' '.join(sentences[i])}")
    print(f"Contains verb: {labels[i]}")

Sentence: Pierre Vinken , 61 years old , will join the board as a nonexecutive director Nov. 29 .
Contains verb: 1
Sentence: Mr. Vinken is chairman of Elsevier N.V. , the Dutch publishing group .
Contains verb: 1
Sentence: Rudolph Agnew , 55 years old and former chairman of Consolidated Gold Fields PLC , was named *-1 a nonexecutive director of this British industrial conglomerate .
Contains verb: 1
Sentence: A form of asbestos once used * * to make Kent cigarette filters has caused a high percentage of cancer deaths among a group of workers exposed * to it more than 30 years ago , researchers reported 0 *T*-1 .
Contains verb: 1
Sentence: The asbestos fiber , crocidolite , is unusually resilient once it enters the lungs , with even brief exposures to it causing symptoms that *T*-1 show up decades later , researchers said 0 *T*-2 .
Contains verb: 1
Sentence: Lorillard Inc. , the unit of New York-based Loews Corp. that *T*-2 makes Kent cigarettes , stopped using crocidolite in its Micron

# Generate Sentence Embeddings
Convert sentences to numerical embeddings using techniques like TF-IDF, Word2Vec, or sentence transformers.

In [ ]:
# Generate Sentence Embeddings

# Initialize the sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2') #lightweight and efficient model

# Convert sentences to embeddings
embeddings = model.encode([' '.join(sentence) for sentence in sentences])

# Display the shape of the embeddings to ensure they are correctly generated
print(f"Shape of embeddings: {embeddings.shape}")

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)

# Convert labels to numpy arrays
y_train = np.array(y_train)
y_test = np.array(y_test)

# Display the shapes of the training and testing sets
print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Testing set shape: {X_test.shape}, {y_test.shape}")

Shape of embeddings: (3914, 384)
Training set shape: (3131, 384), (3131,)
Testing set shape: (783, 384), (783,)


# Prepare Data for Classification
Split the dataset into training and testing sets, and perform any necessary preprocessing.

In [8]:
# Prepare Data for Classification

# Convert labels to numpy arrays if they're not already
y_train = np.array(y_train)
y_test = np.array(y_test)

# Check class balance in training and testing sets
print(f"Class distribution in training set: {np.bincount(y_train)}")
print(f"Class distribution in testing set: {np.bincount(y_test)}")

# Calculate class weights to handle potential imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
print(f"Class weights: {class_weights}")

# Optional: Standardize features if needed
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

Class distribution in training set: [  86 3045]
Class distribution in testing set: [ 20 763]
Class weights: [18.20348837  0.51412151]


# Build Logistic Regression Model
Implement and train a Logistic Regression model to classify sentences based on verb presence.

In [9]:
# Build Logistic Regression Model

# Initialize the Logistic Regression model
log_reg = LogisticRegression(max_iter=1000)

# Train the model on the training data
log_reg.fit(X_train, y_train)

# Predict on the test data
y_pred = log_reg.predict(X_test)

# Calculate the accuracy of the model
accuracy = accuracy_score(y_test, y_pred)

# Display the accuracy
print(f"Accuracy of the Logistic Regression model: {accuracy:.2f}")

Accuracy of the Logistic Regression model: 0.98


# Evaluate Model Performance
Assess the model using metrics such as accuracy, precision, recall, F1-score, and confusion matrix.

In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Calculate precision, recall, and F1-score
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Display precision, recall, and F1-score
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")

# Generate the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Display the confusion matrix
print("Confusion Matrix:")
print(conf_matrix)

Precision: 0.98
Recall: 1.00
F1-score: 0.99
Confusion Matrix:
[[  2  18]
 [  0 763]]


## Results
### Class imbalance
3045 sentences **with** verbs vs 86 **without** verbs

### Poor minority class performance
The cf-matrix shows that the model identified **only** 2/20 sentences **without** verbs in the test set.

### Accuracy seems misleading
The 98% accuracy is **artificially high** by the existing **imbalance** (the model essentially predicts "has verb" for **almost everything**)

## To conclude
The model does not **perform** and **generalize** well